# ADNI

## INIT

In [1]:
from data_model.DataCleaner import DataCleaner, update_variables_support_file
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [6]:
file_codes = ['FUJIREBIOABETA', 'EUROIMMUN']

In [7]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI', 'custom.file_code' : file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())
print(len(zip_files.keys()))


dict_keys(['FUJIREBIOABETA_11Aug2025.csv', 'ADNI_EUROIMMUN_11Aug2025.csv'])
2


# Support file managment
operazione per popolare il file support file per i file considerati

In [8]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

In [9]:
for file_name in list(zip_files.keys()):
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    print(file_code)
    if file_code not in list(support_file['file_code']):
        print('not found in excel')
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

FUJIREBIOABETA
FUJIREBIOABETA
PROBLEMA: nessuna chiave di popolazione trovata
EUROIMMUN
EUROIMMUN
PROBLEMA: nessuna chiave di popolazione trovata


In [10]:
new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)

The ADNI_variables_cleaned1 file has been updated with the new file_code: ['FUJIREBIOABETA']
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned1 file has restored the previous information of the file_code: ['EUROIMMUN']
Open the file and verify it, if needed update the variables names and metadata


## IF SUPPORT FILE already populated

In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)


In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'

Open the new_support_file and fill in the new variable codes.

# FILE SPECIFIC DATA CLEANING 1


## Abeta & Tau in CSF - Elecsys

### UPENNBIOMK_ROCHE_ELECSYS

In [ ]:
file_code = 'UPENNBIOMK_ROCHE_ELECSYS'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'TAU', 'PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df)


In [ ]:
final_df.head()

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UPENNBIOMK_ADNIDIAN_ES_2017

In [ ]:
file_code = 'UPENNBIOMK_ADNIDIAN_ES_2017'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA','TAU','PTAU']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [ ]:
renamed_df.head()

In [ ]:
AbT_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)
final_df, ATN_var = dataCleaner.get_ATN_profile(AbT_df)


In [ ]:
final_df

In [ ]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

In [ ]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Abeta Tau metodo ELISA

### EUROIMMUN

In [11]:
file_code = 'EUROIMMUN'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [12]:
# Important columns
columns_must_be_verified = ['BETA_AMYLOID_1_40', 'BETA_AMYLOID_1_42']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [13]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  120
Adopted visit selection strategy:
 Equal values    119
Name: count, dtype: int64


In [14]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [15]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [16]:
renamed_df.head()

,RID,VISCODE,VISIT_MONTH,EXAMDATE,AB40,AB42,AB4240
0,51,m72,0.0,2011-08-11,7324.79,264.01,0.036043
1,59,m60,0.0,2010-12-15,8431.15,1184.28,0.140465
2,232,m60,0.0,2011-04-05,8454.41,772.85,0.091414
3,259,m60,0.0,2011-04-14,9887.33,580.35,0.058696
4,272,m72,0.0,2012-06-06,9835.15,918.71,0.093411


In [17]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [18]:
final_df

,RID,VISCODE,VISIT_MONTH,EXAMDATE,AB40,AB42,AB4240
0,51,m72,0.0,2011-08-11,7324.79,264.01,0.036043
1,59,m60,0.0,2010-12-15,8431.15,1184.28,0.140465
2,232,m60,0.0,2011-04-05,8454.41,772.85,0.091414
3,259,m60,0.0,2011-04-14,9887.33,580.35,0.058696
4,272,m72,0.0,2012-06-06,9835.15,918.71,0.093411
...,...,...,...,...,...,...,...
154,999999,NaN,NaN,NaT,6176.72,486.86,0.078822
155,999999,NaN,NaN,NaT,6182.38,506.25,0.081886
156,999999,NaN,NaN,NaT,6466.98,453.45,0.070118
157,999999,NaN,NaN,NaT,5606.20,609.93,0.108796


In [19]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

totale subject: 120
subjects with multiple visits:  1


In [20]:

final_df = final_df[final_df['EXAMDATE'].notna()]
final_df

,RID,VISCODE,VISIT_MONTH,EXAMDATE,AB40,AB42,AB4240
0,51,m72,0.0,2011-08-11,7324.79,264.01,0.036043
1,59,m60,0.0,2010-12-15,8431.15,1184.28,0.140465
2,232,m60,0.0,2011-04-05,8454.41,772.85,0.091414
3,259,m60,0.0,2011-04-14,9887.33,580.35,0.058696
4,272,m72,0.0,2012-06-06,9835.15,918.71,0.093411
...,...,...,...,...,...,...,...
114,4823,bl,0.0,2012-07-19,11739.00,844.46,0.071936
115,4928,bl,0.0,2012-09-27,8603.81,600.59,0.069805
116,4964,bl,0.0,2012-10-29,7155.37,401.02,0.056045
117,4987,bl,0.0,2012-12-14,7395.04,796.84,0.107753


In [21]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [22]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [23]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## FUJIREBIOABETA

In [24]:
file_code = 'FUJIREBIOABETA'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ABETA42', 'ABETA40']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, ['EXAMDATE'])

In [29]:
no_none_df

,RID,VISCODE,VISCODE2,EXAMDATE,GUSPECID,VID,DRAWDTE,DER,SITE,VOL,RECDTE,STORDTE,RUN,RUNDATE,ABETA42,ABETA40,ABETA42_40,COMMENTS,update_stamp
0,42,v06,m60,2011-04-14,JA8075FB-05,9.0,2011-04-14,CSF,D023,0.35,2011-04-15,2011-07-22,2,2019-11-21,1022,10468.0,0.098,NaN,2020-02-07 11:12:44.0
1,51,v06,m72,2011-08-11,JA807VJD-04,9.0,2011-08-11,CSF,D099,0.20,2011-11-02,2011-11-03,6,2019-11-21,102,NaN,NaN,Not enough volume for 1-40,2020-02-07 11:12:44.0
2,89,m48,m48,2010-09-28,JA807JJG-04,8.0,2010-09-28,CSF,D073,0.35,2010-09-30,2011-08-25,6,2019-11-21,923,9201.0,0.100,NaN,2020-02-07 11:12:44.0
3,118,m60,m60,2011-02-16,BA8075HL-03,9.0,2011-02-16,CSF,D027,0.35,2011-02-17,2011-07-22,2,2019-11-21,885,9536.0,0.093,NaN,2020-02-07 11:12:44.0
4,126,m60,m60,2011-03-17,JA8075G8-04,9.0,2011-03-17,CSF,D023,0.35,2011-03-18,2011-07-22,2,2019-11-21,487,10189.0,0.048,NaN,2020-02-07 11:12:44.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,5272,v03,bl,2013-09-04,GA80D34P-03,2.0,2013-09-04,CSF,D053,0.35,2013-09-05,2013-12-27,6,2019-11-21,1681,19458.0,0.086,NaN,2020-02-07 11:12:45.0
418,5275,v03,bl,2013-08-27,AA80D23M-03,2.0,2013-08-27,CSF,D135,0.35,2013-08-29,2013-12-19,5,2019-11-26,640,19115.0,0.033,NaN,2020-02-07 11:12:45.0
419,5279,v03,bl,2013-11-18,EA80D3YY-03,2.0,2013-11-18,CSF,D082,0.35,2013-11-19,2014-01-08,6,2019-11-21,847,9576.0,0.088,NaN,2020-02-07 11:12:45.0
420,5292,v03,bl,2013-11-13,EA80D3XT-03,2.0,2013-11-13,CSF,D057,0.35,2013-11-14,2014-01-08,6,2019-11-21,688,13638.0,0.050,NaN,2020-02-07 11:12:45.0


In [30]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [31]:
#Filtering variables
filtered_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE'], prefix='raw')   

In [32]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)


In [33]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)


In [34]:
tot_sub = len(final_df['RID'].unique())
print('totale subject:', tot_sub)
multiple_visists = (final_df['RID'].value_counts() > 1).sum()
print('subjects with multiple visits: ', multiple_visists)

totale subject: 421
subjects with multiple visits:  0


In [35]:
final_df

,RID,VISCODE,VISIT_MONTH,EXAMDATE,AB42,AB40,AB4240
0,42,m60,0,2011-04-14,1022,10468.0,0.097631
1,51,m72,0,2011-08-11,102,NaN,NaN
2,89,m48,0,2010-09-28,923,9201.0,0.100315
3,118,m60,0,2011-02-16,885,9536.0,0.092806
4,126,m60,0,2011-03-17,487,10189.0,0.047797
...,...,...,...,...,...,...,...
416,5272,bl,0,2013-09-04,1681,19458.0,0.086391
417,5275,bl,0,2013-08-27,640,19115.0,0.033482
418,5279,bl,0,2013-11-18,847,9576.0,0.088450
419,5292,bl,0,2013-11-13,688,13638.0,0.050447


In [36]:
new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [37]:
# optaining automatically info to save the file
lst_population = ['ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [38]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## Volumi

### UCSDVOL

In [ ]:
file_code = 'UCSDVOL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['BRAIN', 'EICV', 'VENTRICLES', 'LHIPPOC', 'RHIPPOC']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# Convert QCPASS values from 1/0 to 'complete'/'partial'
no_none_df = dataCleaner.convert_qcpass_values(no_none_df, col_name='QCPASS')
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='QCPASS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSF Longitudinal dataset

In [ ]:
file_codes = ['UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL']
file_code = file_codes[3]

In [ ]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
if file_code == 'UCSFFSL': #the other UCSF longitudinal files have just partial immages segmentation
    no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
if file_code == 'UCSFFSL':
    lst_population = ['ADNI1','ADNIGO','ADNI2']
else:
    lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX 
complete    4485\
partial        1\
hanno solo VISITCODE e non VISITCODE2 inoltre non hanno info sulla popolazione --> da inserire manualmente?

In [ ]:
file_code = 'UCSFFSX' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
######### ERRORE DA RISOLVERE
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFX7

partial     11091\
complete      849

In [ ]:
file_code = 'UCSFFSX7' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') 

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
# update the new info support file
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX6
complete    2222\
partial       18

In [ ]:
file_code = 'UCSFFSX6' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
len(final_df['RID'].value_counts()[final_df['RID'].value_counts() == 1])

### UCSFFSX51

In [ ]:
file_code = 'UCSFFSX51' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has no status column

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

### UCSFFSX51_ADNI1_3T
partial    484


In [ ]:
file_code = 'UCSFFSX51_ADNI1_3T' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# VERIFICARE ma non da usare
test_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## ADNI MERGE

In [ ]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [ ]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['AGE_bl', 'VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE


In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## MMSE

In [ ]:
file_code = 'MMSE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['MMSCORE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## PTDEMOG

In [ ]:
file_code = 'PTDEMOG'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['AGE', 'VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## ADSP_PHC_BIOMARKER

In [ ]:
file_code = 'ADSP_PHC_BIOMARKER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'Tau_RAW', 'pTau_RAW', 'AB42_RAW', 'AT_class']
single_column_required = ['PHC_Diagnosis']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = dataCleaner.binarization_gender(datefix_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df, new_var = dataCleaner.convert_to_dummies_ATNC_profile(processed_df, col_name='AT_class')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [ ]:
new_var

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
processed_df['PHC_Race'] = processed_df['PHC_Race'].map(mapping)

In [ ]:
filtered_df =dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'] + new_var, remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
renamed_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', filtered_df, file_code)

In [ ]:
renamed_df

In [ ]:
final_df, ratios_var = dataCleaner.get_abeta_tau_ratios(renamed_df)

In [ ]:
final_df

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## BLCHANGE

In [ ]:
file_code = 'BLCHANGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='BCPREDX')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## DXSUM

In [ ]:
file_code = 'DXSUM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
        }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='DIAGNOSIS')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)